# 🧠 PCOS × Neurodivergence — Medical RAG Assistant

> **Retrieval-Augmented Generation pipeline grounded in real clinical research papers.**
> Answers questions about the PCOS–neurodivergence connection with cited evidence.

---

### 📌 Pipeline Overview
```
PDFs (research papers)
      ↓
Text Extraction (PyMuPDF)
      ↓
Chunking (RecursiveCharacterTextSplitter)
      ↓
Embeddings (HuggingFace — all-MiniLM-L6-v2)
      ↓
Vector Store (FAISS)
      ↓
Query → Retrieve top-k chunks
      ↓
LLM (Groq — LLaMA 3.3 70B) → Answer with citations
```

---
**Stack:** LangChain · FAISS · HuggingFace Embeddings · Groq (LLaMA 3.3 70B) · PyMuPDF

**Papers included:**
- Cherskov et al. (2018) — PCOS and Autism: prenatal sex steroid theory
- Dubey et al. (2021) — Systematic review: maternal PCOS and neuropsychiatric disorders
- Chen et al. (2020) — Finnish cohort: PCOS and offspring psychiatric disorders
- Redkar & Khan (2025) — PCOS impact on attention
- Berni et al. (2018) — PCOS and adverse mental health/neurodevelopmental outcomes

## ⚙️ Step 1 — Install Dependencies

In [ ]:
!pip install -q langchain langchain-groq langchain-core langchain-text-splitters \
                langchain-huggingface langchain-community \
                faiss-cpu sentence-transformers pymupdf \
                groq tiktoken ipywidgets

## 🔑 Step 2 — Set API Key

Get your free Groq API key from: https://console.groq.com

**Setup:** Click the 🔑 key icon in the left Colab sidebar → Add secret → Name: `GROQ_API_KEY` → paste your key → enable notebook access.

In [ ]:
import os
from google.colab import userdata
from groq import Groq

# Load from Colab Secrets
try:
    os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')
    key = os.environ["GROQ_API_KEY"]
    print(f"✅ Key loaded: {key[:8]}...{key[-4:]} ({len(key)} chars)")
except Exception as e:
    print(f"❌ Could not load from Secrets: {e}")
    print("→ Add GROQ_API_KEY to Colab Secrets (key icon in left sidebar)")

# Verify key works before proceeding
test_client = Groq(api_key=os.environ["GROQ_API_KEY"])
try:
    test_resp = test_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": "say ok"}],
        max_tokens=5
    )
    print(f"✅ API key verified — model responding")
except Exception as e:
    print(f"❌ Key verification failed: {e}")
    print("→ Generate a new key at console.groq.com")

## 📄 Step 3 — Upload Research Papers

Upload your PDF research papers here. Select all 5 files at once.

In [ ]:
from google.colab import files
import os

print("📁 Upload your PDF research papers (select multiple files)...")
uploaded = files.upload()

pdf_dir = "/content/papers/"
os.makedirs(pdf_dir, exist_ok=True)

for filename, content in uploaded.items():
    # Clean filename — remove Colab's " (1)" suffix if re-uploading
    clean_name = filename.replace(' (1)', '').replace(' (2)', '')
    filepath = os.path.join(pdf_dir, clean_name)
    with open(filepath, 'wb') as f:
        f.write(content)

pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]
print(f"\n✅ {len(pdf_files)} PDF(s) ready:")
for f in pdf_files:
    print(f"   📄 {f}")

## 📖 Step 4 — Extract Text from PDFs

In [ ]:
import fitz  # PyMuPDF
from langchain_core.documents import Document

# Friendly paper name mapping for readable citations
PAPER_NAMES = {
    "pmos":  "Cherskov_2018_PCOS_Autism",
    "pmos1": "Redkar_2025_PCOS_Attention",
    "pmos2": "Berni_2018_PCOS_MentalHealth",
    "pmos3": "Dubey_2021_MetaAnalysis_PCOS_NPD",
    "pmos4": "Chen_2020_Finnish_Cohort_PCOS",
}

def extract_text_from_pdfs(pdf_dir):
    """Extract text from all PDFs, preserving source metadata."""
    documents = []
    pdf_files = [f for f in os.listdir(pdf_dir) if f.endswith('.pdf')]

    for filename in pdf_files:
        filepath = os.path.join(pdf_dir, filename)
        doc = fitz.open(filepath)
        raw_name = filename.replace('.pdf', '').replace(' (1)', '').strip()
        # Use friendly name if available, else use filename
        paper_name = PAPER_NAMES.get(raw_name, raw_name)

        for page_num, page in enumerate(doc, start=1):
            text = page.get_text()
            if text.strip():
                documents.append(Document(
                    page_content=text,
                    metadata={
                        "source": paper_name,
                        "page": page_num,
                        "filename": filename
                    }
                ))
        print(f"✅ Extracted {len(doc)} pages from: {filename} → [{paper_name}]")

    return documents

raw_documents = extract_text_from_pdfs(pdf_dir)
print(f"\n📊 Total pages extracted: {len(raw_documents)}")

## ✂️ Step 5 — Chunk Documents

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " "]
)

chunks = splitter.split_documents(raw_documents)

print(f"✅ Total chunks created: {len(chunks)}")
print(f"\n📋 Sample chunk:")
print("-" * 60)
print(f"Source: {chunks[0].metadata['source']} | Page: {chunks[0].metadata['page']}")
print(chunks[0].page_content[:300], "...")

## 🔢 Step 6 — Create Embeddings & FAISS Vector Store

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings  # updated import
from langchain_community.vectorstores import FAISS

print("⏳ Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"}
)

print("⏳ Building FAISS vector store (this may take a minute)...")
vectorstore = FAISS.from_documents(chunks, embeddings)

vectorstore.save_local("/content/faiss_pcos_neuro_index")

print(f"✅ Vector store created with {vectorstore.index.ntotal} vectors")
print("✅ FAISS index saved to /content/faiss_pcos_neuro_index")

## 🤖 Step 7 — Set Up Groq LLM + RAG Chain

In [ ]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# --- LLM ---
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0.1,
    max_tokens=1024,
    groq_api_key=os.environ["GROQ_API_KEY"]
)

# --- Retriever ---
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}
)

# --- Hallucination-aware system prompt ---
SYSTEM_PROMPT = """
You are a medical research assistant specialising in PCOS (Polycystic Ovary Syndrome)
and its connection to neurodivergence (ADHD, autism, cognitive differences).

STRICT RULES:
1. Answer ONLY using the provided research context below.
2. If the context does not contain enough information to answer, say:
   "The uploaded research papers do not contain sufficient information to answer this question."
   Do NOT guess or use outside knowledge.
3. Always cite your source at the end of each claim using: [Source: <paper name>, Page <number>]
4. Keep answers clear, structured, and accessible to both clinicians and patients.
5. Do not make diagnostic claims — present findings as research evidence only.

CONTEXT FROM RESEARCH PAPERS:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", SYSTEM_PROMPT),
    ("human", "{question}")
])

def format_context(docs):
    """Format retrieved chunks with source metadata."""
    formatted = []
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', '?')
        formatted.append(
            f"[Chunk {i} | Source: {source} | Page {page}]\n{doc.page_content}"
        )
    return "\n\n".join(formatted)

# --- RAG Chain ---
rag_chain = (
    {"context": retriever | format_context, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("✅ RAG chain ready!")

## 💬 Step 8 — Query the RAG Assistant

Ask any question about PCOS and neurodivergence — grounded entirely in your uploaded papers.

In [ ]:
def ask(question):
    """Ask the RAG assistant a question and print the answer."""
    print(f"\n❓ Question: {question}")
    print("-" * 70)
    answer = rag_chain.invoke(question)
    print(answer)
    print("-" * 70)
    return answer

ask("What is the link between PCOS and ADHD in women?")

In [ ]:
ask("Are children of mothers with PCOS at higher risk of autism spectrum disorder?")

In [ ]:
ask("How do elevated androgens in PCOS affect brain development?")

In [ ]:
ask("What mental health disorders are most commonly associated with PCOS?")

In [ ]:
ask("Does insulin resistance in PCOS contribute to cognitive impairment?")

In [ ]:
ask("Does obesity in PCOS mothers worsen neuropsychiatric outcomes in children?")

In [ ]:
ask("What did the Finnish cohort study find about maternal PCOS and psychiatric disorders?")

In [ ]:
ask("What is the prenatal sex steroid theory of autism?")

In [ ]:
ask("How does PCOS affect attention and cognitive function in women?")

## 🔍 Step 9 — Inspect Retrieved Chunks (Transparency)

See exactly which paper chunks were used to answer a question.

In [ ]:
def inspect_retrieval(question, k=5):
    """Show which chunks are retrieved for a given question."""
    print(f"\n🔍 Retrieving chunks for: '{question}'")
    print("=" * 70)
    docs = vectorstore.similarity_search(question, k=k)
    for i, doc in enumerate(docs, 1):
        source = doc.metadata.get('source', 'Unknown')
        page = doc.metadata.get('page', '?')
        print(f"\n[Chunk {i}] Source: {source} | Page {page}")
        print("-" * 40)
        print(doc.page_content[:400], "..." if len(doc.page_content) > 400 else "")

inspect_retrieval("What is the link between PCOS and ADHD in women?")

## 💬 Step 10 — Interactive Q&A (Widget)

Type your own questions in the box below and press Enter.

In [ ]:
from ipywidgets import widgets
from IPython.display import display

text = widgets.Text(
    placeholder="Type your question and press Enter...",
    description="Question:",
    layout=widgets.Layout(width='75%')
)
out = widgets.Output()

def on_submit(sender):
    if text.value.strip():
        with out:
            ask(text.value.strip())
        text.value = ""

text.on_submit(on_submit)
print("🧠 PCOS × Neurodivergence RAG Assistant")
print("Type your question below and press Enter:\n")
display(text, out)

---
## 📊 Bonus — Paper Coverage Summary

Which papers are contributing most chunks to the vector store?

In [ ]:
from collections import Counter
import pandas as pd

source_counts = Counter(doc.metadata['source'] for doc in chunks)
df_coverage = pd.DataFrame(
    source_counts.items(),
    columns=['Paper', 'Chunks']
).sort_values('Chunks', ascending=False).reset_index(drop=True)

print("📚 Paper Coverage in Vector Store")
print("=" * 50)
print(df_coverage.to_string(index=False))
print(f"\nTotal chunks: {df_coverage['Chunks'].sum()}")

---

## ✅ Project Summary

| Component | Tool |
|---|---|
| PDF Extraction | PyMuPDF (fitz) |
| Chunking | LangChain RecursiveCharacterTextSplitter |
| Embeddings | HuggingFace all-MiniLM-L6-v2 (langchain-huggingface) |
| Vector Store | FAISS |
| LLM | Groq — LLaMA 3.3 70B Versatile |
| Hallucination Guard | Custom system prompt refusing out-of-context answers |
| Citations | Author name + page number on every answer |
| Interactive UI | ipywidgets (no blocking input loop) |

---
*Built by Preeti Bhardwaj | PCOS × Neurodivergence Research RAG*